# BUS 493 Part 1 explained
## Risk and volatility across five stocks
MSFT, JPM, XOM, COST, and JNJ. Main sample starts January 1, 2016 and ends before November 1, 2026.
Run the cells in order after installing requirements.txt. The supplied snapshot reproduces the results offline.
The PDF explains the findings and limitations. Each script is commented so you can trace the calculation.

We ask which transformation supports stationarity, whether large moves cluster, how risk responds to shocks, and what that implies for the five-stock portfolio.

In [1]:
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "scripts/01_get_data.py").exists():
    raise FileNotFoundError("Open the notebook from the repository root.")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)
TICKER = "MSFT"  # Change to JPM, XOM, COST, or JNJ to explore another stock.


## 1. Prepare the data
Adjusted prices account for dividends and splits. Step 01 rejects duplicate dates, missing/nonpositive prices, and differences in trading dates across stocks. It saves checksums and dates in data/provenance.json.

**Daily log return in percent = 100 × ln(P_today / P_previous_day).**
Percent units give the GARCH optimizer a suitable scale. Decimal return 0.01 means 1%.
Calculate changes before trimming January dates so the first January return uses the prior December price.
Monthly simple returns are also saved for later parts.

In [2]:
run = subprocess.run([sys.executable, str(ROOT / "scripts/01_get_data.py")], check=True, capture_output=True, text=True)
print(run.stdout)
prices = pd.read_csv(ROOT / "data/daily_adjusted_prices.csv", index_col="Date", parse_dates=True)
returns = pd.read_csv(ROOT / "data/daily_log_returns_pct.csv", index_col="Date", parse_dates=True)
print(prices.head().round(3))
print(returns.head().round(4))


Ticker
MSFT    Information Technology
JPM                 Financials
XOM                     Energy
COST          Consumer Staples
JNJ                Health Care
Saved 2723 daily rows, 130 monthly rows; last date 2026-10-30

              MSFT     JPM     XOM     COST     JNJ
Date                                               
2016-01-04  47.681  47.945  49.002  133.457  74.968
2016-01-05  47.898  48.028  49.420  133.783  75.281
2016-01-06  47.028  47.334  49.009  132.545  74.901
2016-01-07  45.393  45.420  48.224  129.508  74.028
2016-01-08  45.532  44.403  47.250  127.242  73.237
              MSFT     JPM     XOM    COST     JNJ
Date                                              
2016-01-04 -1.2333 -3.0495 -0.6306 -1.2210 -2.2048
2016-01-05  0.4552  0.1727  0.8485  0.2442  0.4171
2016-01-06 -1.8332 -1.4541 -0.8355 -0.9297 -0.5067
2016-01-07 -3.5402 -4.1280 -1.6136 -2.3176 -1.1723
2016-01-08  0.3062 -2.2654 -2.0409 -1.7659 -1.0741


### Check one return by hand
The second observation has both prices visible in this table. The first observation uses the prior December close from the source file.

In [3]:
p0, p1 = prices[TICKER].iloc[:2]
calculated = 100 * np.log(p1 / p0)
print(f"Prior price {p0:.5f}; next price {p1:.5f}")
print(f"Calculated return {calculated:.6f}%; saved return {returns[TICKER].iloc[1]:.6f}%")
assert np.isclose(calculated, returns[TICKER].iloc[1])


Prior price 47.68095; next price 47.89848
Calculated return 0.455173%; saved return 0.455173%


## 2. Run all five stocks
The same settings are used for each stock. The script saves numerical evidence in results/. A failed optimizer raises an error. Successful optimization alone does not establish that the model explains all dependence.

In [4]:
run = subprocess.run([sys.executable, str(ROOT / "scripts/02_P1_diagnostics_garch.py")], check=True, capture_output=True, text=True)
print(run.stdout)


MSFT {'persistence': np.float64(0.9688), 'half_life_days': np.float64(21.899), 'sample_annual_vol_pct': np.float64(27.4107), 'long_run_annual_vol_pct': np.float64(28.295)}
JPM {'persistence': np.float64(0.926), 'half_life_days': np.float64(9.0138), 'sample_annual_vol_pct': np.float64(27.158), 'long_run_annual_vol_pct': np.float64(24.9119)}
XOM {'persistence': np.float64(0.9972), 'half_life_days': np.float64(243.1038), 'sample_annual_vol_pct': np.float64(27.8704), 'long_run_annual_vol_pct': np.float64(37.211)}
COST {'persistence': np.float64(0.9438), 'half_life_days': np.float64(11.9885), 'sample_annual_vol_pct': np.float64(21.8414), 'long_run_annual_vol_pct': np.float64(21.528)}
JNJ {'persistence': np.float64(0.9258), 'half_life_days': np.float64(8.9893), 'sample_annual_vol_pct': np.float64(18.6538), 'long_run_annual_vol_pct': np.float64(17.7842)}
Saved complete test tables, fitted series, robust parameter inference, and sensitivity results.



## 3. Stationarity
**Prices → log prices → differences in log prices (returns).**
Taking logs changes the scale but can leave a trend. Differencing measures daily changes.

- ADF starts with a unit-root null. A small p-value rejects it.
- KPSS starts with a stationarity null. A small p-value rejects it.

At 5%, ADF p < 0.05 and KPSS p >= 0.05 jointly support stationarity. Conflicting tests are reported as mixed evidence.
The main tests include an intercept. ADF chooses lags by AIC; KPSS uses automatic bandwidth.
Price/log-price tests also run with a linear trend as a specification check. KPSS p-values at 0.01 and 0.10 are bounds, as indicated by kpss_p_bound.
These tests do not rule out crises, changing tails, or structural breaks.

In [5]:
tests = pd.read_csv(ROOT / "results/stationarity.csv")
columns = ["ticker","series","regression","adf_stat","adf_p","adf_lags","kpss_stat","kpss_p_bound","kpss_p","kpss_lags","decision_5pct"]
print(tests.loc[tests.ticker == TICKER, columns].to_string(index=False))
print("\nAll return-series decisions")
print(tests.loc[tests.series == "log_return_pct", ["ticker","adf_p","kpss_p_bound","kpss_p","decision_5pct"]].to_string(index=False))


ticker         series regression   adf_stat        adf_p  adf_lags  kpss_stat kpss_p_bound  kpss_p  kpss_lags            decision_5pct
  MSFT          price          c  -0.202453 9.382286e-01         0   8.268773           <=    0.01         31 supports nonstationarity
  MSFT      log_price          c  -1.380489 5.915935e-01         9   8.244343           <=    0.01         31 supports nonstationarity
  MSFT log_return_pct          c -17.867048 3.037745e-30         8   0.094267           >=    0.10          5    supports stationarity
  MSFT          price         ct  -3.419316 4.886023e-02         0   0.389559           <=    0.01         31           mixed evidence
  MSFT      log_price         ct  -1.955098 6.256640e-01         9   1.494704           <=    0.01         31 supports nonstationarity

All return-series decisions
ticker        adf_p kpss_p_bound  kpss_p         decision_5pct
  MSFT 3.037745e-30           >=     0.1 supports stationarity
   JPM 4.871854e-28           >=   

## 4. Autocorrelation and volatility clustering
Lag k compares today's observation with the observation k trading days earlier.
Squaring removes the sign and emphasizes large moves. Autocorrelated squared returns indicate clustering in move size even if predicting direction is difficult.
Small return correlations can be statistically detectable with thousands of observations. They do not establish profitable predictability.
The figure bands are approximate white-noise references, not simultaneous confidence intervals valid under all forms of changing variance.

In [6]:
acfs = pd.read_csv(ROOT / "results/acf.csv")
print(acfs[acfs.ticker == TICKER].pivot(index="lag", columns="series", values="acf").round(4))


series  returns  squared_returns
lag                             
1       -0.1241           0.3558
2        0.0059           0.2579
3        0.0052           0.1377
4       -0.0206           0.1401
5        0.0203           0.1502
6       -0.0645           0.1413
7        0.0523           0.1173
8       -0.0964           0.1245
9        0.0889           0.1383
10      -0.0249           0.1669


## 5. GARCH(1,1) with a constant mean
r[t] = mu + epsilon[t]

h[t] = omega + alpha × epsilon[t-1]² + beta × h[t-1]

- mu is average daily percentage log return.
- omega is the daily variance baseline in percent squared.
- alpha is the response to the previous squared surprise.
- beta carries forward the previous conditional variance.
- alpha + beta measures persistence. Near one implies slow decay.

Gaussian innovations define the likelihood; robust standard errors reduce reliance on exact normality for coefficient inference.
A coefficient p-value tests whether that coefficient is zero. Boundary constraints and heavy tails can complicate inference.
The model is symmetric, so equally sized positive and negative surprises affect variance equally.

In [7]:
parameters = pd.read_csv(ROOT / "results/garch_parameters.csv")
print(parameters[parameters.ticker == TICKER].round(6).to_string(index=False))


ticker parameter  estimate  robust_se  p_value    ci_low  ci_high
  MSFT        mu  0.116091   0.028184 0.000038  0.060853 0.171330
  MSFT     omega  0.098984   0.061000 0.104656 -0.020574 0.218542
  MSFT  alpha[1]  0.102146   0.035870 0.004404  0.031842 0.172450
  MSFT   beta[1]  0.866698   0.052772 0.000000  0.763266 0.970130


### Derive the risk measures
For persistence between zero and one,

- Half-life = ln(0.5) / ln(alpha + beta), measured in trading days.
- Long-run daily variance = omega / (1 - alpha - beta), measured in percent squared.
- Long-run annualized volatility = sqrt(long-run daily variance × 252), measured in percent.
- Sample annualized volatility = SD(daily percentage log returns) × sqrt(252).

Compare volatility with volatility. Daily variance has different units.
If persistence reaches one, a finite long-run variance is unavailable. Near one, small parameter changes cause large changes in half-life.

In [8]:
p = parameters[parameters.ticker == TICKER].set_index("parameter").estimate
persistence = p["alpha[1]"] + p["beta[1]"]
if 0 < persistence < 1:
    half_life = np.log(0.5) / np.log(persistence)
    variance = p["omega"] / (1 - persistence)
    print(f"Persistence {persistence:.6f}; half-life {half_life:.2f} trading days")
    print(f"Daily variance {variance:.5f} percent squared; annualized volatility {np.sqrt(252*variance):.2f}%")
else:
    print("Finite long-run risk measures are unavailable.")
risk = pd.read_csv(ROOT / "results/risk_summary.csv").set_index("ticker")
print(risk[["persistence","half_life_days","long_run_variance_pct2","long_run_annual_vol_pct","sample_annual_vol_pct"]].round(4))


Persistence 0.968844; half-life 21.90 trading days
Daily variance 3.17701 percent squared; annualized volatility 28.29%
        persistence  half_life_days  long_run_variance_pct2  long_run_annual_vol_pct  sample_annual_vol_pct
ticker                                                                                                     
MSFT         0.9688         21.8990                  3.1770                  28.2950                27.4107
JPM          0.9260          9.0138                  2.4627                  24.9119                27.1580
XOM          0.9972        243.1038                  5.4947                  37.2110                27.8704
COST         0.9438         11.9885                  1.8391                  21.5280                21.8414
JNJ          0.9258          8.9893                  1.2551                  17.7842                18.6538


## 6. Diagnose limitations and test sensitivity
Standardized residuals divide surprises by fitted daily volatility.
Remaining residual autocorrelation suggests mean dependence. Remaining autocorrelation in squared standardized residuals suggests unmodeled volatility dependence.
The Ljung-Box tests are approximate diagnostics, particularly after fitting parameters and under changing variance.

The 1%/99% capping exercise changes roughly 2% of observations. It asks how sensitive the estimates are to tail observations. The main model retains all observations, including economically meaningful crashes.

In [9]:
diagnostics = pd.read_csv(ROOT / "results/ljung_box.csv")
print(diagnostics.to_string(index=False))
sensitivity = pd.read_csv(ROOT / "results/winsorized_sensitivity.csv").set_index("ticker")
print(risk[["persistence","half_life_days"]].join(sensitivity[["persistence","half_life_days"]], rsuffix="_capped").round(4))


ticker                         series  lag     lb_stat     lb_pvalue
  MSFT                        returns   10  111.962337  2.143989e-19
  MSFT                squared_returns   10  956.333888 4.747691e-199
  MSFT         standardized_residuals   10   15.824921  1.047551e-01
  MSFT squared_standardized_residuals   10    2.872162  9.842744e-01
   JPM                        returns   10  128.743021  8.428940e-23
   JPM                squared_returns   10 2113.316908  0.000000e+00
   JPM         standardized_residuals   10    9.314827  5.025087e-01
   JPM squared_standardized_residuals   10    3.471872  9.680448e-01
   XOM                        returns   10   30.456525  7.210394e-04
   XOM                squared_returns   10 1639.845992  0.000000e+00
   XOM         standardized_residuals   10    2.116059  9.953707e-01
   XOM squared_standardized_residuals   10   20.739112  2.298802e-02
  COST                        returns   10   36.568137  6.719993e-05
  COST                squared_retu

## 7. Regenerate the figures and guide
The scripts read the saved estimates and produce the complete set of Part 1 figures and the explanation PDF.

In [10]:
for name in ["05_figures.py", "part1_report.py"]:
    run = subprocess.run([sys.executable, str(ROOT / "scripts" / name)], check=True, capture_output=True, text=True)
    print(run.stdout)


Saved 16 Part 1 figures in PNG and SVG formats.

Created Part1_Explanation.pdf



![Risk comparison](figures/P1_risk_comparison.png)

## 8. Make the required judgment
Compare sample volatility for the calmer/noisier names, squared-return ACF for clustering, and GARCH half-life for shock duration.
Check the residual diagnostics and sensitivity before treating a ranking as precise.
One stock represents each sector, so these results describe the selected holdings rather than sector-wide averages.
Part 1 motivates a risk posture. Cross-stock covariances and expected returns in Part 3 are needed to choose final weights.
These are full-sample, retrospective estimates and not an out-of-sample trading test.

## Check your understanding
1. Why do small ADF and large KPSS p-values point in the same direction?
2. Why can move size be persistent even when direction is difficult to predict?
3. Why should omega not be directly compared with annual sample volatility?
4. Why is half-life sensitive when persistence is close to one?
5. Which diagnostic would make you cautious about trusting this model?

## Sources
- [ADF](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html)
- [KPSS](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.kpss.html)
- [GARCH](https://arch.readthedocs.io/en/stable/univariate/univariate_volatility_modeling.html)
- Course project instructions, Part 1, page 2. See AI_USE_LOG.txt for AI assistance.
